In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler

# Load dataset
df = pd.read_csv("team_averages_with_FTR.csv")

# Dictionary to store final scaled values
scaled_data = {}

# Convert numeric columns into separate 2D NumPy arrays
for col in df.columns:
    if df[col].dtype in [np.int64, np.float64]:
        globals()[col] = df[col].to_numpy().reshape(-1, 1)

# Initialize scalers
robust_scaler = RobustScaler()
minmax_scaler = MinMaxScaler()

# Apply scaling only to numerical columns
for col in df.columns:
    if df[col].dtype in [np.int64, np.float64]:
        # Apply RobustScaler
        scaled_col_name = f"{col}_robust_scaled"
        globals()[scaled_col_name] = robust_scaler.fit_transform(globals()[col])

        # Apply MinMaxScaler
        final_scaled_col_name = f"{col}_final_scaled"
        globals()[final_scaled_col_name] = minmax_scaler.fit_transform(globals()[scaled_col_name])

        # Store final scaled values in dictionary
        scaled_data[final_scaled_col_name] = globals()[final_scaled_col_name].flatten()

# Include categorical columns (like 'team') in the final DataFrame
for col in df.columns:
    if df[col].dtype == object:
        scaled_data[col] = df[col]

# Convert to DataFrame
final_df = pd.DataFrame(scaled_data)

### 🔥 APPLY OFFENSE WEIGHTED MULTIPLICATION AND SUM 🔥 ###
offense_weights = {
    "FGR_2_final_scaled": 0.332388,
    "FGR_3_final_scaled": 0.062264,
    "FTR_final_scaled": 0.029727,
    "AST_final_scaled": 0.117877,
    "largest_lead_final_scaled": 0.383899,
    "TOV_final_scaled": 0.022009,  # 🔹 TOV will be modified separately (1 - scaled value)
    "OREB_final_scaled": 0.036285,
    "DREB_final_scaled": 0.015551
}

# Store modified rows
modified_values = []

for row in final_df.itertuples(index=False):
    row_updates = {}
    offense_weighted_sum = 0

    for col in final_df.columns:
        if col == "team":
            row_updates[col] = getattr(row, col)
            continue

        if col in offense_weights:
            if col == "TOV_final_scaled":  # 🔹 Modify TOV value
                new_value = (1 - getattr(row, col)) * offense_weights[col]
            else:
                new_value = getattr(row, col) * offense_weights[col]

            offense_weighted_sum += new_value
            row_updates[col] = new_value
        else:
            row_updates[col] = getattr(row, col)

    row_updates["offense_weighted_sum"] = offense_weighted_sum
    modified_values.append(row_updates)

# Convert modified values back to DataFrame
modified_df = pd.DataFrame(modified_values)

### 🔥 SCALE "offense_weighted_sum" TO 0-100 RANGE 🔥 ###
offense_weighted_sum_array = modified_df["offense_weighted_sum"].to_numpy().reshape(-1, 1)
scaled_offense_weighted_sum = MinMaxScaler(feature_range=(0, 100)).fit_transform(offense_weighted_sum_array)
modified_df["offense_weighted_sum_scaled"] = scaled_offense_weighted_sum.flatten()

### 🔥 APPLY DEFENSE WEIGHTED MULTIPLICATION AND SUM 🔥 ###
defense_weights = {
    "DREB_final_scaled": 0.449087,
    "BLK_final_scaled": 0.115509,
    "STL_final_scaled": 0.246018,
    "F_tech_final_scaled": 0.033999,  # 🔴 Negative impact for Technical Fouls
    "F_personal_final_scaled": 0.155386  # 🔴 Negative impact for Personal Fouls
}

def_modified_values = []

for row in final_df.itertuples(index=False):
    def_row_updates = {}
    defense_weighted_sum = 0

    for col in final_df.columns:
        if col == "team":
            def_row_updates[col] = getattr(row, col)
            continue

        if col == "F_personal_final_scaled" or col == "F_tech_final_scaled":
          new_value = ( 1 - getattr(row,col))*defense_weights[col]
        if col in defense_weights:
            new_value = getattr(row, col) * defense_weights[col]
            defense_weighted_sum += new_value
            def_row_updates[col] = new_value
        else:
            def_row_updates[col] = getattr(row, col)

    def_row_updates["defense_weighted_sum"] = defense_weighted_sum
    def_modified_values.append(def_row_updates)

# Convert modified values back to DataFrame
def_modified_df = pd.DataFrame(def_modified_values)

### 🔥 SCALE "defense_weighted_sum" TO 0-100 RANGE 🔥 ###
defense_weighted_sum_array = def_modified_df["defense_weighted_sum"].to_numpy().reshape(-1, 1)
scaled_defense_weighted_sum = MinMaxScaler(feature_range=(0, 100)).fit_transform(defense_weighted_sum_array)
def_modified_df["defense_weighted_sum_scaled"] = scaled_defense_weighted_sum.flatten()

ext_weights = {
    "rest_days_final_scaled" : 0.634394,
    "OT_length_min_tot_final_scaled" : 0.142324,
    "tz_dif_H_E_final_scaled" : 0.111332,
    "prev_game_dist_final_scaled" : 0.069994,
    "home_away_NS_final_scaled" : 0.031916,
    "travel_dist_final_scaled" : 0.010040
}

ext_modified_values = []

for row in final_df.itertuples(index=False):
    ext_row_updates = {}
    ext_weighted_sum = 0

    for col in final_df.columns:
        if col == "team":
            ext_row_updates[col] = getattr(row, col)
            continue

        if col in ext_weights:
            new_value = getattr(row, col) * ext_weights[col]
            ext_weighted_sum += new_value
            ext_row_updates[col] = new_value
        else:
            ext_row_updates[col] = getattr(row, col)

    ext_row_updates["extra_weighted_sum"] = ext_weighted_sum
    ext_modified_values.append(ext_row_updates)

ext_modified_df = pd.DataFrame(ext_modified_values)

extra_weighted_sum_array = ext_modified_df['extra_weighted_sum'].to_numpy().reshape(-1,1)
scaled_ext_rtg = MinMaxScaler(feature_range=(0,100)).fit_transform(extra_weighted_sum_array)
ext_modified_df["extra_weighted_sum_scaled"] = scaled_ext_rtg.flatten()

# Merge offense and defense DataFrames
final_modified_df = modified_df.merge(
    def_modified_df[["team", "defense_weighted_sum", "defense_weighted_sum_scaled"]],
    on="team"
)

done_final_df = final_modified_df.merge(ext_modified_df[["team", "extra_weighted_sum_scaled"]],on = "team")
# Save final modified DataFrame
final_csv_path = "Games22_Scaled_Weighted_100_all3.csv"
done_final_df.to_csv(final_csv_path, index=False)

print(f"Final modified data with weighted sums (offense & defense) scaled to 0-100 saved to '{final_csv_path}' successfully!")

# If in Google Colab, download file
try:
    from google.colab import files
    files.download(final_csv_path)
    print("Download started for:", final_csv_path)
except ImportError:
    print("Not running in Google Colab, skipping download.")

Final modified data with weighted sums (offense & defense) scaled to 0-100 saved to 'Games22_Scaled_Weighted_100_all3.csv' successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started for: Games22_Scaled_Weighted_100_all3.csv


For Attack RTG